**texte en gras**#  ETL Bronze → Silver — Dataset 1 : UCI Online Retail
## Projet 2 — Segmentation RFM, CLV & Optimisation E-Commerce

---

| Champ | Valeur |
|---|---|
| **Dataset** | UCI Online Retail |
| **Volume initial (Bronze)** | ~541 909 lignes × 8 colonnes |
| **Période** | Décembre 2010 – Décembre 2011 |
| **Fichier source** | `online_retail.csv` (UTF-8, ~45 MB) |
| **Sortie Silver** | `online_retail_silver.parquet` |
| **Journal** | `DECISIONS.md` — DEC-001 à DEC-011 |

---

### Règle cible
-  Perte cumulée < **5 %** du volume initial sur `fact_transactions`
- La perte du scope RFM (DEC-004) est documentée et justifiée séparément

## 0. Montage Google Drive & Imports

In [ ]:
# Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# Installation des librairies nécessaires
!pip install ydata-profiling -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.1 MB/s eta 0:00:00


In [1]:
import pandas as pd
import numpy as np
import os
import logging
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration du logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('ETL_UCI')

print('Imports OK')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

Imports OK
   pandas  : 2.2.2
   numpy   : 2.0.2


## 1. Configuration des chemins

In [12]:
BASE_PATH   = '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce'
RAW_PATH    = os.path.join(BASE_PATH, 'Data', 'raw')
SILVER_PATH = os.path.join(BASE_PATH, 'Data', 'silver')

# Création des dossiers si inexistants
print(f"Attempting to create directory: {RAW_PATH}")
os.makedirs(RAW_PATH, exist_ok=True)
if os.path.exists(RAW_PATH):
    print(f"Directory {RAW_PATH} created or already exists.")
else:
    print(f"ERROR: Directory {RAW_PATH} could not be created. Please check Google Drive mount and permissions.")

print(f"Attempting to create directory: {SILVER_PATH}")
os.makedirs(SILVER_PATH, exist_ok=True)
if os.path.exists(SILVER_PATH):
    print(f"Directory {SILVER_PATH} created or already exists.")
else:
    print(f"ERROR: Directory {SILVER_PATH} could not be created.")

# Fichiers
FILE_IN  = os.path.join(RAW_PATH,    'online_retail.csv')
FILE_OUT = os.path.join(SILVER_PATH, 'online_retail_silver.parquet')
RETURNS_OUT   = os.path.join(SILVER_PATH, 'fact_returns.parquet')
ANOMALIES_OUT = os.path.join(SILVER_PATH, 'fact_anomalies_uci.parquet')

print(f'\n Dossier source  : {RAW_PATH}')
print(f' Dossier Silver  : {SILVER_PATH}')
print(f' Fichier source  : {FILE_IN}')
# assert os.path.exists(FILE_IN), f' Fichier introuvable : {FILE_IN}'

Attempting to create directory: /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw
Directory /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw created or already exists.
Attempting to create directory: /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/silver
Directory /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/silver created or already exists.

 Dossier source  : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw
 Dossier Silver  : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/silver
 Fichier source  : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw/online_retail.csv


In [10]:
print(f'\nContenu du dossier RAW_PATH ({RAW_PATH}):')
if os.path.exists(RAW_PATH):
    for item in os.listdir(RAW_PATH):
        print(f'  - {item}')
else:
    print(f'  Le dossier {RAW_PATH} n\'existe pas.')

# You can also manually check if the file exists at FILE_IN
if not os.path.exists(FILE_IN):
    print(f'\n ATTENTION : Le fichier source {FILE_IN} est INTROUVABLE. Veuillez vérifier son chemin et son nom.')
    print('       Assurez-vous que le fichier est bien présent dans votre Google Drive à l\'emplacement indiqué.')


Contenu du dossier RAW_PATH (/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw):
  Le dossier /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw n'existe pas.

 ATTENTION : Le fichier source /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw/online_retail.csv est INTROUVABLE. Veuillez vérifier son chemin et son nom.
       Assurez-vous que le fichier est bien présent dans votre Google Drive à l'emplacement indiqué.


## 2. Chargement Bronze (Ingestion brute)

In [13]:
logger.info('=== CHARGEMENT BRONZE ===')
def load_csv_robust(file_path):
    encodings = ['utf-8', 'ISO-8859-1', 'latin1', 'cp1252']

    for enc in encodings:
        try:
            logger.info(f"Tentative avec encoding: {enc}")

            df = pd.read_csv(
                file_path,
                encoding=enc,
                dtype={
                    'InvoiceNo': str,
                    'StockCode': str,
                    'Description': str,
                    'Quantity': float,
                    'UnitPrice': float,
                    'CustomerID': str,
                    'Country': str
                },
                parse_dates=['InvoiceDate']
            )

            logger.info(f" Succès avec encoding: {enc}")
            return df

        except UnicodeDecodeError:
            logger.warning(f" Échec encoding: {enc}")
            continue

    raise ValueError("Impossible de décoder le fichier avec les encodings testés")

# ── Chargement ──
if not os.path.exists(FILE_IN):
    logger.error(f"Fichier source INTROUVABLE: {FILE_IN}. Veuillez le télécharger manuellement dans le dossier RAW_PATH.")
    raise FileNotFoundError(f"Le fichier '{FILE_IN}' n'existe pas. Veuillez l'uploader manuellement à cet emplacement.")

df_bronze = load_csv_robust(FILE_IN)

# ── Analyse rapide ──
VOLUME_INITIAL = len(df_bronze)
logger.info(f"Volume initial (Bronze) : {VOLUME_INITIAL:,} lignes")

print("\n Aperçu Bronze :")
print(df_bronze.head(3).to_string())

print(f"\n Shape : {df_bronze.shape}")
print(f"\n Types :\n{df_bronze.dtypes}")

ERROR:ETL_UCI:Fichier source INTROUVABLE: /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw/online_retail.csv. Veuillez le télécharger manuellement dans le dossier RAW_PATH.


FileNotFoundError: Le fichier '/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/raw/online_retail.csv' n'existe pas. Veuillez l'uploader manuellement à cet emplacement.

In [ ]:
# Rapport de qualité initiale
print('\n📋 RAPPORT QUALITÉ INITIAL (Bronze)')
print('='*55)
for col in df_bronze.columns:
    n_null = df_bronze[col].isna().sum()
    pct    = n_null / VOLUME_INITIAL * 100
    n_uniq = df_bronze[col].nunique()
    print(f'  {col:<15} | nulls: {n_null:>7,} ({pct:5.1f}%) | uniques: {n_uniq:>6,}')

print(f'\n  Doublons stricts : {df_bronze.duplicated().sum():,}')


📋 RAPPORT QUALITÉ INITIAL (Bronze)
  InvoiceNo       | nulls:       0 (  0.0%) | uniques: 25,900
  StockCode       | nulls:       0 (  0.0%) | uniques:  4,070
  Description     | nulls:   1,454 (  0.3%) | uniques:  4,223
  Quantity        | nulls:       0 (  0.0%) | uniques:    722
  InvoiceDate     | nulls:       0 (  0.0%) | uniques: 23,260
  UnitPrice       | nulls:       0 (  0.0%) | uniques:  1,630
  CustomerID      | nulls: 135,080 ( 24.9%) | uniques:  4,372
  Country         | nulls:       0 (  0.0%) | uniques:     38

  Doublons stricts : 5,268


## 3. Pipeline de Nettoyage Bronze → Silver

### Compteur de pertes (suivi en temps réel)

In [14]:
# Compteur de pertes
losses = {}
df = df_bronze.copy()

def log_loss(dec_id, description, before, after, note=''):
    lost = before - after
    pct  = lost / VOLUME_INITIAL * 100
    losses[dec_id] = {'description': description, 'lost': lost, 'pct': pct}
    logger.info(f'[{dec_id}] {description} → -{lost:,} lignes ({pct:.2f}%)')
    if note:
        logger.info(f'         Note : {note}')
    return after

NameError: name 'df_bronze' is not defined

### DEC-001 — Suppression des doublons stricts

In [ ]:
# ── DEC-001 : Doublons stricts ──────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates(keep='first')
after = len(df)
log_loss('DEC-001', 'Doublons stricts supprimés', before, after,
         'Conservation première occurrence (keep=first)')

print(f'\n  Exemples de doublons (avant suppression — sur Bronze) :')
dupes_example = df_bronze[df_bronze.duplicated(keep=False)].head(4)
if not dupes_example.empty:
    print(dupes_example.to_string())
else:
    print('  (aucun doublon détecté)')


  Exemples de doublons (avant suppression — sur Bronze) :
    InvoiceNo StockCode                    Description  Quantity         InvoiceDate  UnitPrice CustomerID         Country
485    536409     22111   SCOTTIE DOG HOT WATER BOTTLE       1.0 2010-12-01 11:45:00       4.95      17908  United Kingdom
489    536409     22866  HAND WARMER SCOTTY DOG DESIGN       1.0 2010-12-01 11:45:00       2.10      17908  United Kingdom
494    536409     21866    UNION JACK FLAG LUGGAGE TAG       1.0 2010-12-01 11:45:00       1.25      17908  United Kingdom
517    536409     21866    UNION JACK FLAG LUGGAGE TAG       1.0 2010-12-01 11:45:00       1.25      17908  United Kingdom


### DEC-002 — Séparation des factures de retour (préfixe C*)

In [ ]:
# Gestion des retours clients

nb_lignes_avant = len(df)

# Les factures commençant par C correspondent à des retours
retours = df['InvoiceNo'].str.startswith('C', na=False)

# Séparation des ventes et des retours
df_returns = df[retours].copy()
df = df[~retours].copy()

nb_lignes_apres = len(df)

print("\nRetours clients détectés")
print(f"Lignes déplacées vers fact_returns : {len(df_returns):,}")

# Calcul du montant total des retours
df_returns['TotalRevenue'] = df_returns['Quantity'] * df_returns['UnitPrice']

montant_retours = df_returns['TotalRevenue'].sum()

print(f"Montant total des retours : £{montant_retours:,.2f}")
print(f"Nombre de factures retour : {df_returns['InvoiceNo'].nunique():,}")
print(f"Nombre de clients concernés : {df_returns['CustomerID'].nunique():,}")

### DEC-003 — Exclusion des codes produits spéciaux

In [ ]:
# ── DEC-003 : Codes produits spéciaux ───────────────────────────────────────
SPECIAL_CODES = ['POST', 'D', 'C2', 'M', 'BANK CHARGES', 'AMAZONFEE', 'DOT', 'CRUK']

before = len(df)

# Comptage par code avant suppression
print('  Détail des codes spéciaux détectés :')
for code in SPECIAL_CODES:
    count = (df['StockCode'].str.upper() == code.upper()).sum()
    print(f'    {code:<15} : {count:,} lignes')

mask_special = df['StockCode'].str.upper().isin([c.upper() for c in SPECIAL_CODES])
df = df[~mask_special].copy()

after = len(df)
log_loss('DEC-003', 'Codes produits spéciaux exclus', before, after)

  Détail des codes spéciaux détectés :
    POST            : 1,130 lignes
    D               : 0 lignes
    C2              : 142 lignes
    M               : 323 lignes
    BANK CHARGES    : 12 lignes
    AMAZONFEE       : 2 lignes
    DOT             : 709 lignes
    CRUK            : 0 lignes


525072

### DEC-004 — Traitement des CustomerID nuls

In [ ]:
# ── DEC-004 : CustomerID nuls — Stratégie différenciée ──────────────────────
# On conserve TOUTES les lignes dans fact_transactions (pour CA global, ST3)
# On flag uniquement les lignes sans CustomerID

n_null_cid = df['CustomerID'].isna().sum()
pct_null   = n_null_cid / VOLUME_INITIAL * 100

# Ajout du flag
df['is_identified'] = df['CustomerID'].notna()

# CA anonyme
df['TotalRevenue_tmp'] = df['Quantity'] * df['UnitPrice']
ca_anonymous = df.loc[~df['is_identified'], 'TotalRevenue_tmp'].sum()
ca_total     = df['TotalRevenue_tmp'].sum()
df.drop(columns=['TotalRevenue_tmp'], inplace=True)

losses['DEC-004'] = {
    'description': 'CustomerID nuls (flaggés, pas supprimés de fact_transactions)',
    'lost': n_null_cid,
    'pct': pct_null
}

logger.info(f'[DEC-004] CustomerID nuls : {n_null_cid:,} ({pct_null:.1f}%) — '
            f'conservés dans fact_transactions, exclus du scope RFM')
print(f'\n    Lignes anonymes          : {n_null_cid:,} ({pct_null:.1f}%)')
print(f'   CA anonyme estimé         : £{ca_anonymous:,.2f}')
print(f'   % CA anonyme / CA total   : {ca_anonymous/ca_total*100:.1f}%')
print(f'\n   Stratégie : is_identified=False → exclus de mart_rfm uniquement')


  🏷️  Lignes anonymes          : 133,885 (24.7%)
  💰 CA anonyme estimé         : £1,499,678.48
  📊 % CA anonyme / CA total   : 14.6%

  ✅ Stratégie : is_identified=False → exclus de mart_rfm uniquement


### DEC-005 — Exclusion des UnitPrice nuls ou négatifs

In [ ]:
# ── DEC-005 : UnitPrice <= 0 ─────────────────────────────────────────────────
before = len(df)

n_zero_price = (df['UnitPrice'] == 0).sum()
n_neg_price  = (df['UnitPrice'] < 0).sum()
print(f'  UnitPrice = 0  : {n_zero_price:,} lignes')
print(f'  UnitPrice < 0  : {n_neg_price:,} lignes')

df = df[df['UnitPrice'] > 0].copy()
after = len(df)
log_loss('DEC-005', 'UnitPrice <= 0 exclus', before, after)

  UnitPrice = 0  : 2,496 lignes
  UnitPrice < 0  : 2 lignes


522574

### DEC-006 — Quantity négatives hors factures C*

In [ ]:
# ── DEC-006 : Quantity < 0 sans préfixe C ───────────────────────────────────
before = len(df)

mask_neg_qty = (df['Quantity'] < 0) & (~df['InvoiceNo'].str.startswith('C', na=False))
df_anomalies = df[mask_neg_qty].copy()
df_anomalies['anomaly_type'] = 'negative_qty_without_C_prefix'
df = df[~mask_neg_qty].copy()

after = len(df)
log_loss('DEC-006', 'Quantity < 0 hors C* → fact_anomalies', before, after)

print(f'\n   {len(df_anomalies)} anomalie(s) loggée(s) dans fact_anomalies')


  🚨 0 anomalie(s) loggée(s) dans fact_anomalies


### DEC-007 — Quantity = 0

In [ ]:
# ── DEC-007 : Quantity = 0 ──────────────────────────────────────────────────
before = len(df)
df = df[df['Quantity'] != 0].copy()
after = len(df)
log_loss('DEC-007', 'Quantity = 0 exclus', before, after)

522574

### DEC-008 — Standardisation Description

In [ ]:
# ── DEC-008 : Normalisation Description ─────────────────────────────────────
import re

# 1. UPPER CASE + strip
df['Description'] = df['Description'].str.upper().str.strip()

# 2. Nettoyage caractères spéciaux non ASCII
df['Description'] = df['Description'].apply(
    lambda x: re.sub(r'[^\x20-\x7E]', '', str(x)) if pd.notna(x) else x
)

# 3. Imputation des nulls par la description la plus fréquente du StockCode
desc_mode = (
    df.dropna(subset=['Description'])
      .groupby('StockCode')['Description']
      .agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else 'DESCRIPTION INCONNUE')
)
n_null_desc_before = df['Description'].isna().sum()
df['Description'] = df.apply(
    lambda row: desc_mode.get(row['StockCode'], 'DESCRIPTION INCONNUE')
    if pd.isna(row['Description']) else row['Description'],
    axis=1
)
n_null_desc_after = df['Description'].isna().sum()

losses['DEC-008'] = {'description': 'Description normalisée (0 ligne supprimée)', 'lost': 0, 'pct': 0.0}
logger.info(f'[DEC-008] Description normalisée — {n_null_desc_before} nulls imputés')
print(f'  Nulls Description avant imputation : {n_null_desc_before:,}')
print(f'  Nulls Description après imputation : {n_null_desc_after:,}')

### DEC-009 — Traitement des outliers extrêmes de Quantity

In [ ]:
# ── DEC-009 : Outliers Quantity (IQR × 3) ───────────────────────────────────
Q1 = df['Quantity'].quantile(0.25)
Q3 = df['Quantity'].quantile(0.75)
IQR = Q3 - Q1
LOWER_BOUND = Q1 - 3 * IQR
UPPER_BOUND = Q3 + 3 * IQR

n_outliers = ((df['Quantity'] < LOWER_BOUND) | (df['Quantity'] > UPPER_BOUND)).sum()

# Flag uniquement — pas de suppression (commandes grossistes légitimes)
df['is_quantity_outlier'] = (
    (df['Quantity'] < LOWER_BOUND) | (df['Quantity'] > UPPER_BOUND)
)

losses['DEC-009'] = {'description': 'Outliers Quantity flaggés (non supprimés)', 'lost': 0, 'pct': 0.0}
logger.info(f'[DEC-009] Outliers Quantity flaggés : {n_outliers:,} (seuil IQR×3: [{LOWER_BOUND:.0f}, {UPPER_BOUND:.0f}])')

print(f'  Q1={Q1:.0f} | Q3={Q3:.0f} | IQR={IQR:.0f}')
print(f'  Seuil bas  : {LOWER_BOUND:.0f}')
print(f'  Seuil haut : {UPPER_BOUND:.0f}')
print(f'  Outliers flaggés : {n_outliers:,} lignes')
print(f'   Stratégie : flag is_quantity_outlier=True (conservation)')

### DEC-010 — Parsing et enrichissement InvoiceDate

In [ ]:
# ── DEC-010 : Feature Engineering temporel ─────────────────────────────────

# Vérification du parsing (déjà fait au chargement)
n_null_date = df['InvoiceDate'].isna().sum()
if n_null_date > 0:
    df = df.dropna(subset=['InvoiceDate'])
    logger.info(f'[DEC-010] {n_null_date} dates nulles supprimées')

# Enrichissement temporel
df['invoice_year']       = df['InvoiceDate'].dt.year
df['invoice_month']      = df['InvoiceDate'].dt.month
df['invoice_week']       = df['InvoiceDate'].dt.isocalendar().week.astype(int)
df['invoice_day_of_week']= df['InvoiceDate'].dt.dayofweek   # 0=Lundi
df['invoice_hour']       = df['InvoiceDate'].dt.hour
df['invoice_date_only']  = df['InvoiceDate'].dt.date

losses['DEC-010'] = {'description': 'Feature Engineering temporel (0 ligne supprimée)', 'lost': 0, 'pct': 0.0}
logger.info('[DEC-010] Variables temporelles créées')

print('   Variables créées :')
for col in ['invoice_year','invoice_month','invoice_week','invoice_day_of_week','invoice_hour','invoice_date_only']:
    print(f'    + {col}')

print(f'\n  Période couverte : {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')

### DEC-011 — Calcul de TotalRevenue et CustomerID nettoyé

In [ ]:
# ── DEC-011 : Calcul TotalRevenue + nettoyage CustomerID ─────────────────────

# TotalRevenue
df['TotalRevenue'] = (df['Quantity'] * df['UnitPrice']).round(2)

# CustomerID : nettoyage (supprimer .0 si float importé)
df['CustomerID'] = df['CustomerID'].apply(
    lambda x: str(int(float(x))) if pd.notna(x) and str(x).replace('.0','').isdigit() else x
)

# Flag is_return dans fact_transactions (toujours False ici, mais utile pour joins)
df['is_return'] = False

losses['DEC-011'] = {'description': 'TotalRevenue calculé + CustomerID normalisé (0 perte)', 'lost': 0, 'pct': 0.0}
logger.info('[DEC-011] TotalRevenue calculé, CustomerID normalisé')

print(f'   TotalRevenue ajouté')
print(f'   CA Silver total : £{df["TotalRevenue"].sum():,.2f}')
print(f'   Panier moyen    : £{df.groupby("InvoiceNo")["TotalRevenue"].sum().mean():,.2f}')

## 4. Bilan Cumulé des Pertes

In [ ]:
print('\n' + '='*65)
print('  BILAN CUMULÉ DES PERTES — UCI Online Retail')
print('='*65)
print(f'  Volume initial (Bronze) : {VOLUME_INITIAL:>10,}')
print()

total_lost_fact = 0
for dec_id, info in losses.items():
    marker = ' ' if dec_id == 'DEC-004' else '  '
    print(f'{marker}{dec_id} | {info["description"]:<45} | -{info["lost"]:>7,} ({info["pct"]:5.2f}%)')
    if dec_id != 'DEC-004' and info['lost'] > 0:
        total_lost_fact += info['lost']

pct_fact = total_lost_fact / VOLUME_INITIAL * 100
pct_rfm  = (total_lost_fact + losses.get('DEC-004',{}).get('lost',0)) / VOLUME_INITIAL * 100

print()
print(f'  Volume Silver fact_transactions : {len(df):>10,}')
print(f'  Volume fact_returns             : {len(df_returns):>10,}')
print()
print(f'  Perte fact_transactions (hors DEC-004) : {pct_fact:.2f}% '
      + ('' if pct_fact < 5 else '❌'))
print(f'  Perte scope RFM (avec DEC-004)         : {pct_rfm:.1f}%   documentée')
print('='*65)

## 5. Schéma final Silver

In [ ]:
# Réorganisation des colonnes Silver
SILVER_COLS = [
    'InvoiceNo', 'InvoiceDate', 'invoice_year', 'invoice_month',
    'invoice_week', 'invoice_day_of_week', 'invoice_hour', 'invoice_date_only',
    'CustomerID', 'is_identified',
    'StockCode', 'Description',
    'Quantity', 'UnitPrice', 'TotalRevenue',
    'Country',
    'is_return', 'is_quantity_outlier'
]

df_silver = df[SILVER_COLS].copy()

print(' Schéma Silver final :')
print(df_silver.dtypes.to_string())
print(f'\n Shape Silver : {df_silver.shape}')
print(f'\n  Nulls restants :')
null_remaining = df_silver.isna().sum()
print(null_remaining[null_remaining > 0].to_string() if null_remaining.sum() > 0 else '  Aucun null critique')

## 6. Sauvegarde Parquet

In [ ]:
# Sauvegarde fact_transactions Silver
df_silver.to_parquet(FILE_OUT, index=False, engine='pyarrow')
logger.info(f' fact_transactions sauvegardé : {FILE_OUT}')
print(f'   fact_transactions → {FILE_OUT}')
print(f'     {len(df_silver):,} lignes | {os.path.getsize(FILE_OUT)/1024/1024:.1f} MB')

# Sauvegarde fact_returns
df_returns.to_parquet(RETURNS_OUT, index=False, engine='pyarrow')
logger.info(f' fact_returns sauvegardé : {RETURNS_OUT}')
print(f'   fact_returns       → {RETURNS_OUT}')
print(f'     {len(df_returns):,} lignes')

# Sauvegarde anomalies
if len(df_anomalies) > 0:
    df_anomalies.to_parquet(ANOMALIES_OUT, index=False, engine='pyarrow')
    print(f'   fact_anomalies     → {ANOMALIES_OUT}')
    print(f'     {len(df_anomalies):,} lignes')

## 7. Vérification de reproductibilité

In [ ]:
# Vérification — rechargement du fichier Silver
df_check = pd.read_parquet(FILE_OUT)

assert len(df_check) == len(df_silver), 'Erreur de reproductibilité : volumes différents'
assert list(df_check.columns) == list(df_silver.columns), ' Erreur de reproductibilité : colonnes différentes'

print(' Vérification de reproductibilité : OK')
print(f'   {len(df_check):,} lignes relues depuis le Parquet')
print(f'   Colonnes : {list(df_check.columns)}')

## 8. Statistiques finales Silver

In [ ]:
print('\n STATISTIQUES SILVER — UCI Online Retail')
print('='*55)
print(f'  Transactions valides  : {len(df_silver):,}')
print(f'  Clients identifiés    : {df_silver["CustomerID"].nunique():,}')
print(f'  SKUs distincts        : {df_silver["StockCode"].nunique():,}')
print(f'  Factures distinctes   : {df_silver["InvoiceNo"].nunique():,}')
print(f'  Pays couverts         : {df_silver["Country"].nunique():,}')
print(f'  CA total Silver       : £{df_silver["TotalRevenue"].sum():,.2f}')
print(f'  Panier moyen          : £{df_silver.groupby("InvoiceNo")["TotalRevenue"].sum().mean():,.2f}')
print(f'  Période               : {df_silver["InvoiceDate"].min().date()} → {df_silver["InvoiceDate"].max().date()}')
print()
print(f'  Transactions avec CustomerID (scope RFM) : {df_silver["is_identified"].sum():,}')
print(f'  Transactions anonymes (ST3 uniquement)   : {(~df_silver["is_identified"]).sum():,}')

In [8]:
from datetime import datetime

print(f'\n Pipeline ETL Bronze → Silver UCI terminé avec succès.')
print(f'   {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'\n   Fichiers produits :')
print(f'   → {FILE_OUT}')
print(f'   → {RETURNS_OUT}')
if 'df_anomalies' in locals() and len(df_anomalies) > 0:
    print(f'   → {ANOMALIES_OUT}')


✅ Pipeline ETL Bronze → Silver UCI terminé avec succès.
   2026-06-15 07:27:41

   Fichiers produits :
   → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/silver/online_retail_silver.parquet
   → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/silver/fact_returns.parquet
